<a href="https://colab.research.google.com/github/chaseshallow/ansci_4040_final_class_project/blob/main/Step05bPoolFull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 05b · Pool the fingerprints: full

Collects the **full fingerprints from Step 04b** of every video folder in `CalvingPenPipelineOutputs/` into one **pool**, and labels every track with the cow it shows.

**Labels:** each video folder gets an `Identities.csv` (TrackId, IdentityId, Use, Note). On the first run every track is set to the video's one cow, e.g. `Pen2_20260926_Cow01`. Open it in Drive / Google Sheets to change a track's cow, or set `Use` to `False` to leave a track out (e.g. the calf). This file is **never overwritten**; new tracks are added, edits are kept.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/Pool/`, **overwriting** each run:

- `PoolEmbeddings.npy` + `PoolIndex.csv`: all fingerprints with their cow label (near-duplicate crops left out)
- `PoolSummary.csv`: per cow, how many videos, tracks and crops
- `SelfCheck.csv` / `.png`: each crop's closest crop in **another** track (or its own track more than 5 min away), as if it were new: how well a cow finds herself

**After switching the tracker to SAM (Step 02d), delete the video's `Identities.csv` once** before running this: its TrackIds came from ByteTrack and no longer match.

No GPU needed. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/ansci_4040_final_class_project.git"
RepoBranch = "main"
RepoDir = "/content/ansci_4040_final_class_project"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Pool import BuildPool
from CalvingPipeline.Paths import OutputRoot

In [ ]:
# 2. Settings: the only cell to edit.
EmbeddingFolder = "Step04bFull"   # which step 04 output to pool, in every video folder
PoolDir = OutputRoot / "Pool"
VideoStems = None                 # None = every video folder that has Step04bFull; or a list of folder names
SkipNearDuplicates = True         # leave out crops almost identical to the previous one of their track
ExcludeSeconds = 300              # self-check: a crop may also match its own track if > 5 min apart (SAM tracks are long)

In [ ]:
# 3. Run: build the pool, then show the summary and the self-check.
import pandas as pd
from IPython.display import Image, display

Stats = BuildPool(OutputRoot, PoolDir, EmbeddingFolder=EmbeddingFolder, VideoStems=VideoStems,
                  SkipNearDuplicates=SkipNearDuplicates, ExcludeSeconds=ExcludeSeconds)

display(pd.read_csv(PoolDir / "PoolSummary.csv"))
Check = pd.read_csv(PoolDir / "SelfCheck.csv")
if len(Check):
    display(Image(str(PoolDir / "SelfCheck.png")))
    # Per track: how close its crops get to the same cow in other tracks (higher = easier to recognise).
    display(Check.groupby(["VideoStem", "TrackId", "IdentityId"])
                 .agg(Crops=("CropPath", "count"), MedianBestSameCow=("BestSameCow", "median"),
                      MinBestSameCow=("BestSameCow", "min"), Correct=("Correct", "mean"))
                 .round(3).reset_index())

## About this step

**Model:** none; it reuses the step 04 fingerprints (all videos must come from the same model, or it stops with an error).

**Self-check:** for every crop it hides that crop's own track and looks for the most similar crop in the rest of the pool, as if the track were new. `BestSameCow` = best similarity to the same cow, `BestOtherCow` = best similarity to a different cow, `Correct` = the overall best match is the same cow. With one cow in the pool only `BestSameCow` exists; once a second cow is added, the two histograms in `SelfCheck.png` show whether (and where) a threshold separates them.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.